# 10-714 第四次作业

在这次作业中，你将利用前三次作业中搭建的所有组件，使用高性能网络结构解决一些现代问题。首先，我们将利用新的CPU/CUDA后端添加几个新操作。接着，你将实现卷积操作，以及一个卷积神经网络，用于在CIFAR-10图像分类数据集上训练分类器。然后，你将实现循环神经网络（RNN）和长短期记忆网络（LSTM），并在宾州树库数据集上进行词级别预测的语言建模。

和往常一样，我们将从复制此笔记本并获取起始代码开始。
提醒：__你必须在云端硬盘中保存一份副本__。

In [ ]:
# Code to set up the assignment
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/
!mkdir -p 10714
%cd /content/drive/MyDrive/10714
!git clone https://github.com/dlsys10714/hw4.git
%cd /content/drive/MyDrive/10714/hw4

!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git
!pip3 install pybind11

In [ ]:
# REQUIRED FOR MUGRADE
MY_API_KEY = "<FILL YOUR API KEY HERE>"
HW4_NAME = "hw4"

In [ ]:
!make
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

In [1]:
import sys
sys.path.append('./python')

# Download the datasets you will be using for this assignment

import urllib.request
import os

!mkdir -p './data/ptb'
# Download Penn Treebank dataset
ptb_data = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb."
for f in ['train.txt', 'test.txt', 'valid.txt']:
    if not os.path.exists(os.path.join('./data/ptb', f)):
        urllib.request.urlretrieve(ptb_data + f, os.path.join('./data/ptb', f))

# Download CIFAR-10 dataset
if not os.path.isdir("./data/cifar-10-batches-py"):
    urllib.request.urlretrieve("https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz", "./data/cifar-10-python.tar.gz")
    !tar -xvzf './data/cifar-10-python.tar.gz' -C './data'

6597.40s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


KeyboardInterrupt: 

为了完成本次作业的初始设置，请参考你上次作业的解答代码，将 `python/needle/autograd.py` 文件中所有的代码填充完整。同时，请将第三次作业中的解答代码复制到 `src/ndarray_backend_cpu.cc` 和 `src/ndarray_backend_cuda.cu` 这两个文件中。

**注意**：在复制前面作业的代码时，请小心操作，不要意外删除或修改任何新的导入语句和函数声明。

## 第一部分：ND后端 [10分]

回顾一下，在第二次作业中，`array_api`是作为`numpy`导入的。在本部分中，目标是使用从needle后端`NDArray`（位于`python/needle/backend_ndarray/ndarray.py`中）导入的`array_api`来编写必要的操作。请确保从第三次作业中复制`reshape`、`permute`、`broadcast_to`和`__getitem__`的解答代码。

请在`python/needle/ops/ops_logarithmic.py`和`python/needle/ops/ops_mathematic.py`文件中填写以下类：

- `PowerScalar`
- `EWiseDiv`
- `DivScalar`
- `Transpose`
- `Reshape`
- `BroadcastTo`
- `Summation`
- `MatMul`
- `Negate`
- `Log`
- `Exp`
- `ReLU`
- `LogSumExp`
- `Tanh`（新增）
- `Stack`（新增）
- `Split`（新增）

请注意，对于其中的大部分操作，你在之前的作业中已经编写过解答代码，因此不应更改之前解答的大部分内容。如果出现问题，请检查所使用的`array_api`函数是否在needle后端中得到支持。

`Tanh`、`Stack`和`Split`是新增的操作符。`Stack`沿新轴连接大小相同的张量，而`Split`则撤销此操作。这两个操作的梯度可以相互用对方的形式表示。我们不直接对`Split`进行测试，仅测试`Stack`的反向传播（我们假定你在实现`Stack`时使用了`Split`）。

**注意：** 你可能希望让你的`Summation`操作支持对多个轴进行求和；如果你的`BroadcastTo`操作支持同时沿多个轴进行广播，那么在进行该操作的反向传播时，你可能需要这个功能。但这一点更多是出于便利性而非必要性，我们将此决定权留给你（没有相应的测试）。

**注意：** 根据你的实现方式，在重塑数组之前，你可能需要确保调用`.compact()`方法。（如果这是必要的，你将在后续的作业中遇到相应的错误信息。）

**注意：** 在从之前的作业复制代码时，请小心不要意外删除或修改任何新的导入语句和函数声明。

In [ ]:
!python3 -m pytest -l -v -k "nd_backend"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW4_NAME" -k "new_nd_backend"

## 第二部分：CIFAR-10数据集 [10分]

接下来，你需要编写对[CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html)图像分类数据集的支持。该数据集包含60,000张32x32的彩色图像，分为10个类别，每个类别有6,000张图像。其中训练集有50,000张图像，测试集有10,000张图像。

首先，在`python/needle/data/datasets/cifar10_dataset.py`文件中的`CIFAR10Dataset`类里实现`__init__`函数。你可以通过上面的链接了解如何正确读取你在作业开始时下载的CIFAR-10数据集文件。同时，请填写`__getitem__`和`__len__`方法。注意，`__getitem__`返回的数据形状应为(3, 32, 32)的顺序。

从之前的作业中复制`python/needle/data/data_transforms.py`和`python/needle/data/data_basic.py`文件。

In [ ]:
!python3 -m pytest -l -v -k "test_cifar10"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW4_NAME" -k "cifar10"

## 第三部分：卷积神经网络 [40分]

以下是此任务中你需要完成的工作概述。

在 `python/needle/backend_ndarray/ndarray.py` 中实现：
- `flip`
- `pad`

在 `python/needle/ops/ops_mathematic.py` 中实现（前向和反向传播）：
- `Flip`
- `Dilate`
- `UnDilate`
- `Conv`

在 `python/needle/nn/nn_conv.py` 中实现：
- `Conv`

在 `apps/models.py` 中，填充 `ResNet9` 类。

在 `apps/simple_ml.py` 中，填充以下函数：
- `epoch_general_cifar10`
- `train_cifar10`
- `evaluate_cifar10`

我们已在 `python/needle/nn/nn_basic.py` 中为你提供了 `BatchNorm2d` 的实现，作为对之前 `BatchNorm1d` 实现的封装。

**注意**：请记住从之前的作业中复制 `nn_basic.py` 的解答代码，同时确保不要覆盖 `BatchNorm2d` 模块。

### 填充 ndarrays

深度学习库中通常实现的卷积会减小输入的尺寸；
例如，一张 (1, 32, 32, 3) 的图像与一个 3x3 的滤波器进行卷积，会得到一个 (1, 30, 30, c) 的输出。
解决这一问题的方法是在执行卷积之前对输入 ndarray 进行填充，例如用零填充以获得 (1, 34, 34, 3) 的 ndarray，从而使结果变为 (1, 32, 32, 3)。

卷积的反向传播也需要进行填充操作。

你需要在 `ndarray.py` 中实现 `pad` 函数，使其紧密贴合 `np.pad` 的行为。
也就是说，`pad` 函数应接收一个由二元组组成的元组，该元组的长度等于数组的维度数，
其中每个二元组分别对应"左填充"和"右填充"。

例如，如果 `A` 是一个 (10, 32, 32, 8) 的 ndarray（可视为 NHWC 格式），那么 `A.pad( (0, 0), (2, 2), (2, 2), (0, 0) )` 将得到一个 (10, 36, 36, 8) 的 ndarray，其中"空间"维度在所有边上都填充了两个零。

In [ ]:
!python3 -m pytest -l -v -k "pad_forward"

### Flipping ndarrays & FlipOp

In [ ]:
import numpy as np
import ctypes

以下是用于下面演示的一些辅助代码，你大概率可以忽略这些代码。不过，查看一下 `offset` 函数可能会有所启发。

In [ ]:
# reads off the underlying data array in order (i.e., offset 0, offset 1, ..., offset n)
# i.e., ignoring strides
def raw_data(X):
    X = np.array(X) # copy, thus compact X
    return np.frombuffer(ctypes.string_at(X.ctypes.data, X.nbytes), dtype=X.dtype, count=X.size)

# Xold and Xnew should reference the same underlying data
def offset(Xold, Xnew):
    assert Xold.itemsize == Xnew.itemsize
    # compare addresses to the beginning of the arrays
    return (Xnew.ctypes.data - Xold.ctypes.data)//Xnew.itemsize

def strides(X):
    return ', '.join([str(x//X.itemsize) for x in X.strides])

def format_array(X, shape):
    assert len(shape) == 3, "I only made this formatting work for ndims = 3"
    def chunks(l, n):
        n = max(1, n)
        return (l[i:i+n] for i in range(0, len(l), n))
    a = [str(x) if x >= 10 else ' ' + str(x) for x in X]
    a = ['(' + ' '.join(y) + ')' for y in [x for x in chunks(a, shape[-1])]]
    a = ['|' + ' '.join(y) + '|' for y in [x for x in chunks(a, shape[-2])]]
    return '  '.join(a)

def inspect_array(X, *, is_a_copy_of):
    # compacts X, then reads it off in order
    print('Data: %s' % format_array(raw_data(X), X.shape))
    # compares address of X to copy_of, thus finding X's offset
    print('Offset: %s' % offset(is_a_copy_of, X))
    print('Strides: %s' % strides(X))

为了实现二维卷积的反向传播，我们（可能）需要一个能够翻转 ndarray 坐标轴的函数。我们之所以说"可能"，是因为你或许可以通过巧妙实现卷积前向函数来避免使用该函数。不过，我们认为如果可以沿着卷积核的垂直和水平维度进行"翻转"，这个问题会更容易理解。

下面我们将尝试帮助你建立对"翻转"操作的直观理解，以便你在 `ndarray.py` 中实现该功能。为此，我们将探讨 numpy 的 `np.flip` 函数。需要注意的是，`flip` 通常通过使用负步长并改变底层数组的_偏移量_来实现。

例如，在数组的_所有_坐标轴上翻转数组等效于反转该数组。在这种情况下，可以想象我们需要所有步长都为负值，并且偏移量设置为数组的长度（从数组末尾开始，并反向"步进"）。

由于在上次作业的实现中我们没有显式支持负步长，我们只需使用这些步长调用 `NDArray.make` 来创建"翻转"后的数组，然后立即调用 `.compact()`。除了在少数地方将无符号整数改为有符号整数外，我们推测现有的 `compact` 函数无需任何修改即可支持负步长。在我们分发的 `.cc` 和 `.cu` 文件中，我们已经修改了函数签名以反映这一点。

或者，你也可以通过在 CPU 后端复制内存的方式简单地实现 `flip`，这种方式可能更直观。我们建议你按照下面的迷你教程进行操作，将实现重点放在 Python 部分，因为我们相信即使以稍显原始的方式在 C 语言中实现该功能，所需的工作量也大致相同。

使用以下数组作为其他示例的参考：

In [ ]:
A = np.arange(1, 25).reshape(3, 2, 4)
inspect_array(A, is_a_copy_of=A)

我们已在数组的每个坐标轴周围加上了方括号。请注意，对于该数组，偏移量为 0，且所有步长均为正值。

---

请观察下方沿最后一个坐标轴翻转数组时发生的变化。
请注意，`inspect_array` 函数会在翻转数组后对其进行压缩，以便你查看数据的"逻辑"顺序，而偏移量则是通过比较**未**压缩的翻转数组与 `is_copy_of`（即我们上文查看的数组 `A`）的地址计算得出的。

也就是说，我们正在观察 numpy 如何为翻转后的数组计算步长和偏移量，以便在我们的实现中复制这一行为。

In [ ]:
inspect_array(np.flip(A, (2,)), is_a_copy_of=A)

因此，翻转最后一个坐标轴会逆转每个四维"单元"内元素的顺序，如上所示。被翻转坐标轴对应的步长已取负值。偏移量变为 3 —— 这很好理解，例如，我们希望新数组的"第一个"元素是 4，而它在 `A` 中的索引正是 3。

In [ ]:
inspect_array(np.flip(A, (1,)), is_a_copy_of=A)

同样地，对于中间坐标轴：我们将中间步长取负，偏移量为 4，这似乎很合理，因为现在我们希望第一个元素是 5，而它在原始数组 `A` 中的索引正是 4。

In [ ]:
inspect_array(np.flip(A, (0,)), is_a_copy_of=A)

尝试推断出在给定翻转坐标轴时计算偏移量的更通用算法。

---

观察当我们翻转_所有_坐标轴时会发生什么。

In [ ]:
inspect_array(np.flip(A, (0, 1, 2)), is_a_copy_of=A)

如前所述，此时偏移量足以指向数组的最后一个元素，而这正是 `A` 的"逆序"版本。

当我们只翻转第 1 轴和第 0 轴时……

In [ ]:
inspect_array(np.flip(A, (0, 1)), is_a_copy_of=A)

偏移量为 20。回顾我们之前对偏移量的计算，你是否注意到了什么？

---

通过探索 numpy 的 ndarray 翻转功能（该功能使用负步长和自定义偏移量），尝试在 `ndarray.py` 中实现 `flip`。同时，你还需要在 `ops_mathematic.py` 中实现"flip"的前向和后向函数；请注意，这些函数应该非常简短。

**重要提示：** 你应该使用新的步长和偏移量调用 NDArray.make，然后立即对该数组执行 `.compact()`。得到的数组会被复制并具有正步长。我们希望采用这种（非最优的）行为，因为我们在之前的实现中没有考虑负步长。_附注：_ 如果你想进一步思考，请考虑负步长在你的实现中可能会在哪些地方/情况下导致问题。`__getitem__` 由于我们处理切片的方式而肯定无法正常工作；还有其他问题吗？（_注：_ 这部分不计入评分。）

另外，如果你希望在 CPU/CUDA 后端添加 `flip` 运算符的实现，这也是可以的。

In [ ]:
!python3 -m pytest -l -v -k "flip"

---

膨胀运算符会在 ndarray 的元素之间插入零。当卷积步长大于1时，我们需要用它来计算卷积的反向传播。举例来说，对于一个 2x2 矩阵，当在两个轴上都膨胀 1 时，膨胀操作应产生如下结果：

$$
\begin{bmatrix}
1 & 2 \\
3 & 4
\end{bmatrix}
\Longrightarrow
\begin{bmatrix}
1 & 0 & 2 & 0 \\
0 & 0 & 0 & 0 \\
3 & 0 & 4 & 0 \\
0 & 0 & 0 & 0
\end{bmatrix}
$$

为了理解为什么步长卷积的反向传播需要膨胀，考虑一个 `stride=2`、`padding="same"`、`input_channels=output_channels=8` 的卷积，作用于大小为 (10, 32, 32, 8) 的输入。由于步长的原因，输出的大小将为 (10, 16, 16, 8)，因此 `out_grad` 的形状将是 (10, 16, 16, 8)。然而，输入的梯度当然需要具有 (10, 32, 32, 8) 的形状——所以我们必须以某种方式增大 `out_grad` 的尺寸。另外请注意，你也可以将步长卷积实现为 `Conv(x)[:, ::2, ::2, :]`，即仅保留空间维度中每隔一个像素。

请在 `ops_mathematic.py` 中实现 `Dilate` 和 `UnDilate`。每个运算符接受两个额外参数（在 attrs 中）：`dilation`（膨胀量）和要膨胀的 `axes`（坐标轴）。你还必须实现相应的操作 `UnDilate`，其前向传播将用于实现 `Dilate` 的梯度（这样我们就无需实现 `GetItem` 和 `SetItem` 操作，因为如果没有额外优化，这些操作的反向传播效率极低）。

**注意**：膨胀量是可加的，而不是乘性的。在上面的例子中，膨胀量为 `1` 意味着在膨胀的每个坐标轴上，每个元素之间都添加一行/一列零（对于未膨胀的每个坐标轴则移除一行/一列）。膨胀量为 `0` 表示没有变化。

In [ ]:
!python3 -m pytest -l -v -k "dilate"

### 提交新的操作（flip/dilation）到 mugrade [10 分]

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW4_NAME" -k "new_ops"

---

### 卷积前向传播

在 `ops_mathematic.py` 中实现二维多通道卷积的前向传播。建议参考课程中的[这个 notebook](https://github.com/dlsyscourse/public_notebooks/blob/main/convolution_implementation.ipynb)，它使用 numpy 中的 im2col 方法实现了二维多通道卷积。

**注意：** 你的卷积操作应接受 NHWC 格式的张量（如上例所示），以及格式为 (kernel_size, kernel_size, input_channels, output_channels) 的权重。

但是，你需要额外添加两个功能。你的卷积函数应该接受 `padding`（默认值为 0）和 `stride`（默认值为 1）这两个参数。对于 `padding`，你只需将填充函数应用于空间维度（即轴 1 和轴 2）。

实现步长卷积只需要对普通卷积实现进行相对较少的修改。

我们建议逐步实现完整的功能集：先实现在没有步长情况下的卷积，确保通过下面的部分测试，然后再添加对步长的支持。

In [ ]:
!python3 -m pytest -l -v -k "op_conv and forward"

---

### 寻找二维多通道卷积的梯度

从技术角度来看，寻找二维多通道卷积的梯度可能相当具有挑战性（尤其是"严格"证明）。这里我们提供一些有用的提示。基本上，我们鼓励你利用一个令人惊讶的事实：_只要维度能够匹配，通常就是正确的方向_。

最终，卷积的反向传播可以通过卷积算子本身来实现，只需对参数和结果巧妙地运用 `flip`、`dilate` 和多次 `transpose` 操作。

在上一节中，我们本质上将卷积实现为矩阵乘积：忽略各种重排和重塑操作，我们基本得到类似 `X @ W` 的形式，其中 `X` 是输入，`W` 是权重。我们还有 `out_grad`，其形状与 `X @ W` 相同。现在，你已经在之前的作业中实现了矩阵乘法的反向传播，我们可以利用这些知识来理解卷积的反向传播。特别地，参考你的矩阵乘法反向传播实现，你可能会注意到（这里只是启发式说明）：

`X.grad = out_grad @ W.transpose` \
`W.grad = X.transpose @ out_grad`

令人惊讶的是，如果我们假设这些也都是卷积操作（现在假设 `out_grad`、`W` 和 `X` 是适用于二维多通道卷积的张量而非矩阵），那么事情就迎刃而解了：

`X.grad ≈ ≈conv(≈out_grad, ≈W)` \
`W.grad ≈ ≈conv(≈X, ≈out_grad)`

其中"≈"表示你需要对这些项应用一些额外的算子，以使维度匹配，例如置换/转置轴、进行膨胀、更改卷积函数的 `padding=` 参数，或对卷积结果进行置换/转置轴操作。

正如我们在课堂上的[最后几张幻灯片](https://dlsyscourse.org/slides/conv_nets.pdf)中所看到的，转置卷积只需通过翻转卷积核即可实现。由于我们处理的是二维而非一维，这意味着需要在垂直和水平方向上翻转卷积核（这就是我们实现 `flip` 的原因）。

以下是关于 `X.grad` 和 `W.grad` 的一些提示总结：

**X.grad**
- 对 `out_grad` 和 `W` 进行卷积，并对它们应用一些操作
- `W` 应沿两个卷积核维度进行翻转
- 如果卷积有步长，则通过相应的膨胀来增大 `out_grad` 的大小
- 通过示例分析维度：注意你想要的 `X.grad` 的形状，并思考如何置换/转置参数以及向卷积添加填充来实现这个形状
    - 这个填充取决于卷积核大小和卷积的 `padding` 参数

**W.grad**
- 对 `X` 和 `out_grad` 进行卷积，并对它们应用一些操作
- `W` 的梯度必须在批次上进行累积；如何让卷积算子自身完成这种累积？
    - 考虑通过转置/置换将批次转化为通道
- 分析维度：如何修改 `X` 和 `out_grad`，使它们卷积后的形状与 `W` 的形状匹配？你可能需要对结果进行转置/置换
    - 记住要考虑传递给卷积的 `padding` 参数

**通用建议**
- 最后处理步长卷积（当你通过大部分测试后，应该能够直接使用 `dilate`）
- 从 `padding=0` 的情况开始，然后考虑更改 `padding` 参数
- 可以通过多次调用 `transpose` 来"置换"轴

提前跳到 nn.Conv 部分，通过前向测试，然后结合下面的测试和 nn.Conv 的反向测试来调试你的实现，这可能会很有帮助。

In [ ]:
!python3 -m pytest -l -v -k "op_conv and backward"

### nn.Conv

#### 修复卷积操作的 `_calculate_fans` 初始化

此前，我们实现了 Kaiming 均匀分布/正态分布初始化，其中我们基本上将 `fan_in = input_size` 和 `fan_out = output_size`。对于卷积操作，这需要更详细的处理，即你需要将这两个值都乘以"感受野大小"，在本例中感受野大小就是卷积核尺寸的乘积——由于我们的卷积核始终是相同大小的 $k\times k$ 核。

**你需要编辑 `python/needle/init/init_initializers.py` 中的 `kaiming_uniform` 及其他初始化函数，使其支持多维数组。** 具体来说，它应该支持一个新增的 `shape` 参数，该参数将被传递给底层的 `rand` 函数等。也就是说，如果 `shape` 参数不是 `None`，则忽略 `fan_in` 和 `fan_out`，直接使用 `shape` 的值来进行初始化。

你可以在下方测试这一点；虽然它不会_直接_计入评分，但它必须与我们的实现相匹配，才能通过 nn.Conv 的 mugrade 测试。

In [ ]:
!python3 -m pytest -l -v -k "kaiming_uniform"

#### 实现 nn.Conv

本质上，nn.Conv 只是我们之前实现的卷积算子的一个封装器，它增加了偏置项，初始化了权重和偏置，并确保设置适当的填充以使输入和输出维度相同（至少在 `stride=1` 的情况下如此）。

重要的是，nn.Conv 应该支持 NCHW 格式而非 NHWC 格式。具体来说，考虑到我们当前的 BatchNorm 实现，我们认为 NCHW 格式更合理。你可以通过对输入和输出各应用两次 `transpose` 来实现这一点。

- 确保 nn.Conv 能够处理 `(N, C, H, W)` 格式的张量，尽管我们为 `(N, H, W, C)` 格式的张量实现了卷积算子
- 使用默认设置的 Kaiming 均匀初始化来初始化 `(k, k, i, o)` 权重张量
- 使用区间 $\displaystyle\pm\frac{1}{\sqrt{\verb|in_channels| \times \verb|kernel_size|^2}}$ 的均匀初始化来初始化 `(o,)` 偏置张量
- 计算适当的填充以确保输入和输出维度相同
- 计算卷积，然后如果存在偏置项，则添加适当广播后的偏置项

你现在可以通过下面两个 PyTest 调用来测试你的 nn.Conv 与 PyTorch 的 nn.Conv2d 的匹配情况。

In [ ]:
!python3 -m pytest -l -v -k "nn_conv_forward"

In [ ]:
!python3 -m pytest -l -v -k "nn_conv_backward"

---

### Implementing "ResNet9"

#### 使用卷积层实现类似 ResNet9 的模型

现在你将使用你的卷积层来实现一个类似 _ResNet9_ 的模型，该模型被认为是能在 CIFAR-10 上快速获得较好精度的合理模型（参见[此处](https://github.com/davidcpage/cifar10-fast)）。我们的主要改动是：使用步长卷积代替池化，并将所有通道数除以 4 以提高性能（因为我们的框架不如工业级框架优化得好）。

在下图中，在第一个线性层之前，你应该对张量进行"展平"操作。你可以使用 `nn_basic.py` 中的 `Flatten` 模块，也可以在 ResNet9 的 `forward()` 方法中直接使用 `.reshape` 来实现。

请确保将设备传递给你的模型中的所有模块；否则，在尝试使用 CUDA 运行时，你将遇到设备不匹配的错误。

<center><img src="https://github.com/dlsyscourse/hw4/blob/main/ResNet9.png?raw=true" alt="ResNet9" style="width: 400px;" /></center>

我们已尽力使本作业的测试比之前实现模型的作业更容易通过。具体来说，我们只需要确保模型具有正确的参数数量，以及在处理 1-2 个批次的 CIFAR-10 数据后具有相似的准确度和损失。

In [ ]:
!python3 -m pytest -l -v -k "resnet9"

现在我们可以开始在 CIFAR10 上训练 ResNet 模型了
（请记得从之前的作业中复制 python/needle/optim.py 中的解决方案）

In [ ]:
!python3 -m pytest -l -v -k "train_cifar10"

### Submit ResNet9 to mugrade [10 points]

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW4_NAME" -k "resnet9"

现在，你可以使用以下代码在 CIFAR-10 上训练你的模型。请注意，由于缺乏数据增强，这一过程可能会相当缓慢，且准确率也不会特别高。你预计每个 epoch 大约需要 500 秒。

In [ ]:
import sys
sys.path.append('./python')
sys.path.append('./apps')
import needle as ndl
from models import ResNet9
from simple_ml import train_cifar10, evaluate_cifar10

device = ndl.cpu()
dataset = ndl.data.CIFAR10Dataset("data/cifar-10-batches-py", train=True)
dataloader = ndl.data.DataLoader(\
         dataset=dataset,
         batch_size=128,
         shuffle=True,)
model = ResNet9(device=device, dtype="float32")
train_cifar10(model, dataloader, n_epochs=10, optimizer=ndl.optim.Adam,
      lr=0.001, weight_decay=0.001)
evaluate_cifar10(model, dataloader)

## 第四部分：循环神经网络 [10分]

**注意：** 在接下来的部分中，你可能会想要对张量进行索引操作，即使用 getitem 或 setitem。然而，我们尚未在我们的库中为张量实现这些操作；相反，你应该使用 `stack` 和 `split` 操作。

在 `python/needle/nn/nn_sequence.py` 中，实现 `RNNCell`。

$h^\prime = \text{tanh}(xW_{ih} + b_{ih} + hW_{hh} + b_{hh})$。如果非线性函数是 'relu'，则使用 ReLU 代替 tanh。

所有权重和偏置应在区间 $\displaystyle\pm\frac{1}{\sqrt{\verb|hidden_size|}}$ 内进行均匀初始化。

在 `python/needle/nn/nn_sequence.py` 中，实现 `RNN`。

对于输入序列中的每个元素，每一层计算以下函数：

$h_t = \text{tanh}(x_tW_{ih} + b_{ih} + h_{(t-1)}W_{hh} + b_{hh})$

其中 $h_t$ 是时间步 $t$ 的隐藏状态，$x_t$ 是时间步 $t$ 的输入，$h_{(t-1)}$ 是前一层在时间步 $t-1$ 的隐藏状态，或时间步 $0$ 的初始隐藏状态。如果非线性函数是 'relu'，则使用 ReLU 代替 tanh。

在多层的 RNN 中，第 $l$ 层（$l \ge 2$）的输入 $x_t^{(l)}$ 是前一层（即第 $l-1$ 层）的隐藏状态 $h_t^{(l-1)}$。

In [ ]:
!python3 -m pytest -l -v -k "test_rnn"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW4_NAME" -k "rnn"

## 第五部分：长短期记忆网络 [10分]

在 `python/needle/nn/nn_sequence.py` 中，实现 `Sigmoid`。

$$\sigma(x) = \frac{1}{1 + \text{exp}(-x)}$$

在 `python/needle/nn/nn_sequence.py` 中，实现 `LSTMCell`。

\begin{align*}
i &= \sigma(xW_{ii} + b_{ii} + hW_{hi} + b_{hi}) \\
f &= \sigma(xW_{if} + b_{if} + hW_{hf} + b_{hf}) \\
g &= \text{tanh}(xW_{ig} + b_{ig} + hW_{hg} + b_{hg}) \\
o &= \sigma(xW_{io} + b_{io} + hW_{ho} + b_{ho}) \\
c^\prime &= f * c + i * g \\
h^\prime &= o * \text{tanh}(c^\prime)
\end{align*}

其中 $\sigma$ 是 sigmoid 函数，而 $i$、$f$、$g$、$o$ 分别代表输入门、遗忘门、细胞门和输出门。

所有权重和偏置应在区间 $\displaystyle\pm\frac{1}{\sqrt{\verb|hidden_size|}}$ 内进行均匀初始化。

现在，在 `python/needle/nn/nn_sequence.py` 中实现 `LSTM`，其功能是将一个多层 LSTM RNN 应用于输入序列。对于输入序列中的每个元素，每一层计算以下函数：

$$\begin{align*}
i_t &= \sigma(x_tW_{ii} + b_{ii} + h_{(t-1)}W_{hi} + b_{hi}) \\
f_t &= \sigma(x_tW_{if} + b_{if} + h_{(t-1)}W_{hf} + b_{hf}) \\
g_t &= \text{tanh}(x_tW_{ig} + b_{ig} + h_{(t-1)}W_{hg} + b_{hg}) \\
o_t &= \sigma(x_tW_{io} + b_{io} + h_{(t-1)}W_{ho} + b_{ho}) \\
c_t &= f * c_{(t-1)} + i * g \\
h_t &= o * \text{tanh}(c_t)
\end{align*}$$

其中 $h_t$ 是时间步 $t$ 的隐藏状态，$c_t$ 是时间步 $t$ 的细胞状态，$x_t$ 是时间步 $t$ 的输入，$h_{(t-1)}$ 是该层在时间步 $t-1$ 的隐藏状态或时间步 $0$ 的初始隐藏状态，而 $i_t$、$f_t$、$g_t$、$o_t$ 分别代表时间步 $t$ 的输入门、遗忘门、细胞门和输出门。

在多层的 LSTM 中，第 $l$ 层（$l \ge 2$）的输入 $x_t^{(l)}$ 是前一层（即第 $l-1$ 层）的隐藏状态 $h_t^{(l-1)}$。

In [ ]:
!python3 -m pytest -l -v -k "test_lstm"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW4_NAME" -k "lstm"

## 第六部分：宾州树库数据集 [10分]

在词级语言建模任务中，模型根据序列中已观察到的单词来预测下一个单词的概率。你将编写对宾州树库数据集的支持，该数据集由《华尔街日报》的故事组成，用于训练和评估基于词级预测的语言模型。

在 `python/needle/data/datasets/ptb_dataset.py` 中，首先实现 `Dictionary` 类，该类从单词列表创建字典，将每个单词映射到一个唯一的整数。

接下来，我们将使用这个 `Dictionary` 类，从你在笔记本开头下载的宾州树库数据集的训练和测试文本文件中创建语料库。实现 `Corpus` 类中的 `tokenize` 函数来完成此操作。

为了准备用于训练和评估的数据，你接下来需要实现 `batchify` 函数。从序列数据开始，batchify 将数据集排列成列。例如，以字母序列作为输入且批大小为 4，我们会得到：

```
┌ a g m s ┐
│ b h n t │
│ c i o u │
│ d j p v │
│ e k q w │
└ f l r x ┘
```

模型将这些列视为独立的，这意味着例如 'g' 对 'f' 的依赖关系无法被学习，但这允许更高效的批处理。

接下来，实现 `get_batch` 函数。`get_batch` 将源数据细分为长度为 `bptt` 的块。如果源数据等于 batchify 函数的示例输出，且 bptt 限制为 2，则当 i = 0 时，我们会得到以下两个 `Tensor`：

```
┌ a g m s ┐ ┌ b h n t ┐
└ b h n t ┘ └ c i o u ┘
```

请注意，尽管函数名称如此，数据的细分并不是沿着批次维度（即维度 1）进行的，因为这部分已由 batchify 函数处理。这些块是沿着维度 0 划分的，对应于 LSTM 或 RNN 中的 seq_len 维度。此外，根据函数文档，返回的第二个 `Tensor`（目标值）应被重塑为一维。

In [ ]:
!python3 -m pytest -l -v -k "ptb"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW4_NAME" -k "ptb"

## 第七部分：训练词级语言模型 [10分]

最后，你将使用已编写的 `RNN` 和 `LSTM` 组件来构建一个语言模型，并在宾州树库数据集上进行训练。

首先，在 `python/needle/nn/nn_sequence.py` 中实现 `Embedding`。假设我们有一个包含1000个单词的字典。那么对于一个索引指向该字典的单词，我们可以将其表示为一个大小为1000的独热向量，然后通过一个线性层将该向量投影到某个嵌入大小的向量。

在 `apps/models.py` 中，你现在可以实现 `LanguageModel`。你的语言模型应包含以下组件：

- 一个嵌入层（将单词ID映射为词嵌入）
- 一个序列模型（RNN或LSTM）
- 一个线性层（输出下一个单词的概率）

在 `apps/simple_ml.py` 中实现 `epoch_general_ptb`、`train_ptb` 和 `evaluate_ptb`。

In [ ]:
!python3 -m pytest -l -v -k "language_model_implementation"

In [ ]:
!python3 -m pytest -l -v -k "language_model_training"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW4_NAME" -k "language_model"

Now, you can train your language model on the Penn Treebank dataset:

In [ ]:
import needle as ndl
sys.path.append('./apps')
from models import LanguageModel
from simple_ml import train_ptb, evaluate_ptb

device = ndl.cpu()
corpus = ndl.data.Corpus("data/ptb")
train_data = ndl.data.batchify(corpus.train, batch_size=16, device=ndl.cpu(), dtype="float32")
model = LanguageModel(30, len(corpus.dictionary), hidden_size=10, num_layers=2, seq_model='rnn', device=ndl.cpu())
train_ptb(model, train_data, seq_len=1, n_epochs=1, device=device)
evaluate_ptb(model, train_data, seq_len=40, device=device)